In [1]:
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter


# ============================================================
# SETTINGS
# ============================================================

PRED_FILE = "pred4.csv"
EVAL_FILE = "eval4.csv"
OUTPUT_FILE = "trait_evaluation.xlsx"

KEY_COL = "key"
TAXON_COL = "taxon"

# Columns which are metadata rather than traits
META_COLS = {
    "key",
    "taxon",
    "source",
    "source_id",
    "description",
    "text",
    "Unnamed: 0",
}

# Optional canonicalisation of categorical values.
# Add other known equivalent forms here if useful.
CANONICAL_MAP = {
    "cypselae": "cypsela",
}

UNIT_TO_METRES = {
    "mm": 0.001,
    "cm": 0.01,
    "m": 1.0,
}


# ============================================================
# NORMALISATION
# ============================================================

def is_empty(value):
    """Return True for blank/NaN values."""
    return pd.isna(value) or str(value).strip() == ""


def canonicalise_text(value):
    """Normalise one categorical trait value."""
    value = str(value).strip().lower()

    # Collapse repeated whitespace
    value = re.sub(r"\s+", " ", value)

    # Optional canonical mapping
    value = CANONICAL_MAP.get(value, value)

    return value


def get_column_unit(column):
    """
    Extract dimensional unit from a column name.

    Examples:
        plant min. height [m] -> m
        seed max. length [mm] -> mm
    """
    match = re.search(r"\[(mm|cm|m)\]", column.lower())

    if match:
        return match.group(1)

    return None


def normalise_measurement(token, target_unit):
    """
    Convert measurement(s) to the unit specified by the column.

    Examples for a [m] column:
        "0.06 m" -> 0.06
        "0.06"   -> 0.06
        "6 cm"   -> 0.06
    """

    token = str(token).strip().lower()

    # Get all numbers
    numbers = re.findall(r"-?\d+(?:\.\d+)?", token)

    if not numbers:
        return set()

    # Detect explicit unit in value
    unit_match = re.search(r"\b(mm|cm|m)\b", token)

    if unit_match:
        source_unit = unit_match.group(1)
    else:
        # If no unit is specified, assume the column's unit
        source_unit = target_unit

    result = set()

    for number in numbers:
        number = float(number)

        # Convert via metres
        metres = number * UNIT_TO_METRES[source_unit]
        converted = metres / UNIT_TO_METRES[target_unit]

        # Round to prevent floating point mismatches
        result.add(round(converted, 10))

    return result


def normalise_cell(column, value):
    """
    Convert a cell into a set of individual normalised values.

    This allows:
        "ovate, lanceolate"
    to be compared with:
        "lanceolate, ovate"
    """

    if is_empty(value):
        return set()

    unit = get_column_unit(column)

    # Dimensional field
    if unit:
        return normalise_measurement(value, unit)

    # Categorical field
    values = str(value).split(",")

    return {
        canonicalise_text(v)
        for v in values
        if v.strip()
    }


# ============================================================
# COMPARISON
# ============================================================

def evaluate_traits(pred_df, eval_df):
    """
    Compare all matching rows and trait values.

    Returns:
        row_summary
        trait_summary
        differences
        mismatch_cells
    """

    # Only compare columns present in both files
    trait_cols = [
        col
        for col in pred_df.columns.intersection(eval_df.columns)
        if col not in META_COLS
        and not col.startswith("Unnamed:")
    ]

    # Index by key
    pred_indexed = pred_df.set_index(KEY_COL, drop=False)
    eval_indexed = eval_df.set_index(KEY_COL, drop=False)

    shared_keys = [
        key
        for key in eval_df[KEY_COL]
        if key in pred_indexed.index
    ]

    row_results = []
    differences = []
    mismatch_cells = set()

    # Store column-level totals
    trait_totals = {
        trait: {
            "matched": 0,
            "extra_pred": 0,
            "missed_eval": 0,
            "exact_cells": 0,
            "different_cells": 0,
            "populated_cells": 0,
        }
        for trait in trait_cols
    }

    for key in shared_keys:

        pred_row = pred_indexed.loc[key]
        eval_row = eval_indexed.loc[key]

        # Protect against duplicate keys
        if isinstance(pred_row, pd.DataFrame):
            pred_row = pred_row.iloc[0]

        if isinstance(eval_row, pd.DataFrame):
            eval_row = eval_row.iloc[0]

        taxon = (
            eval_row[TAXON_COL]
            if TAXON_COL in eval_row.index
            else key
        )

        source = (
            eval_row["source"]
            if "source" in eval_row.index
            else ""
        )

        row_matched = 0
        row_extra = 0
        row_missed = 0

        exact_trait_cells = 0
        different_trait_cells = 0
        populated_trait_cells = 0

        for trait in trait_cols:

            pred_values = normalise_cell(
                trait,
                pred_row.get(trait)
            )

            eval_values = normalise_cell(
                trait,
                eval_row.get(trait)
            )

            # Don't include empty-empty cells in accuracy
            if not pred_values and not eval_values:
                continue

            populated_trait_cells += 1
            trait_totals[trait]["populated_cells"] += 1

            matched = pred_values & eval_values
            extra_pred = pred_values - eval_values
            missed_eval = eval_values - pred_values

            n_matched = len(matched)
            n_extra = len(extra_pred)
            n_missed = len(missed_eval)

            row_matched += n_matched
            row_extra += n_extra
            row_missed += n_missed

            trait_totals[trait]["matched"] += n_matched
            trait_totals[trait]["extra_pred"] += n_extra
            trait_totals[trait]["missed_eval"] += n_missed

            # Exact cell match
            if pred_values == eval_values:
                exact_trait_cells += 1
                trait_totals[trait]["exact_cells"] += 1

            else:
                different_trait_cells += 1
                trait_totals[trait]["different_cells"] += 1

                mismatch_cells.add((key, trait))

                differences.append({
                    "taxon": taxon,
                    "source": source,
                    "key": key,
                    "trait": trait,

                    "Pred": pred_row.get(trait),
                    "Eval": eval_row.get(trait),

                    "matched values": ", ".join(
                        map(str, sorted(matched))
                    ),

                    "extra predicted values": ", ".join(
                        map(str, sorted(extra_pred))
                    ),

                    "missed eval values": ", ".join(
                        map(str, sorted(missed_eval))
                    ),

                    "matched count": n_matched,
                    "extra pred count": n_extra,
                    "missed eval count": n_missed,
                })

        unmatched = row_extra + row_missed
        total_union = row_matched + unmatched

        value_accuracy = (
            100 * row_matched / total_union
            if total_union
            else 100
        )

        cell_accuracy = (
            100 * exact_trait_cells / populated_trait_cells
            if populated_trait_cells
            else 100
        )

        row_results.append({
            "taxon": taxon,
            "source": source,
            "key": key,

            "matched values": row_matched,
            "extra predicted values": row_extra,
            "missed eval values": row_missed,
            "unmatched values": unmatched,
            "total values": total_union,

            "value accuracy %": value_accuracy,

            "exact trait cells": exact_trait_cells,
            "different trait cells": different_trait_cells,
            "populated trait cells": populated_trait_cells,
            "trait cell accuracy %": cell_accuracy,
        })

    # --------------------------------------------------------
    # Trait summaries
    # --------------------------------------------------------

    trait_results = []

    for trait, counts in trait_totals.items():

        matched = counts["matched"]
        extra = counts["extra_pred"]
        missed = counts["missed_eval"]

        unmatched = extra + missed
        total = matched + unmatched

        # Ignore traits absent from every comparable record
        if counts["populated_cells"] == 0:
            continue

        value_accuracy = (
            100 * matched / total
            if total
            else 100
        )

        cell_accuracy = (
            100
            * counts["exact_cells"]
            / counts["populated_cells"]
        )

        trait_results.append({
            "trait": trait,

            "matched values": matched,
            "extra predicted values": extra,
            "missed eval values": missed,
            "unmatched values": unmatched,
            "total values": total,

            "value accuracy %": value_accuracy,

            "exact cells": counts["exact_cells"],
            "different cells": counts["different_cells"],
            "populated cells": counts["populated_cells"],
            "trait cell accuracy %": cell_accuracy,
        })

    row_summary = pd.DataFrame(row_results)
    trait_summary = pd.DataFrame(trait_results)
    differences_df = pd.DataFrame(differences)

    return (
        row_summary,
        trait_summary,
        differences_df,
        mismatch_cells,
    )


# ============================================================
# EXCEL OUTPUT
# ============================================================

def create_excel_report(
    pred_df,
    eval_df,
    row_summary,
    trait_summary,
    differences,
    mismatch_cells,
    output_file,
):

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        pred_df.to_excel(
            writer,
            sheet_name="Pred highlighted",
            index=False
        )

        eval_df.to_excel(
            writer,
            sheet_name="Eval highlighted",
            index=False
        )

        row_summary.to_excel(
            writer,
            sheet_name="Row summary",
            index=False
        )

        trait_summary.to_excel(
            writer,
            sheet_name="Trait summary",
            index=False
        )

        differences.to_excel(
            writer,
            sheet_name="Differences",
            index=False
        )

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    wb = load_workbook(output_file)

    red_fill = PatternFill(
        fill_type="solid",
        fgColor="FFC7CE",
    )

    red_font = Font(
        color="9C0006"
    )

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    header_font = Font(
        bold=True
    )

    for sheet_name, dataframe in [
        ("Pred highlighted", pred_df),
        ("Eval highlighted", eval_df),
    ]:

        ws = wb[sheet_name]

        # Map headers to Excel columns
        headers = {
            cell.value: cell.column
            for cell in ws[1]
        }

        # Key -> Excel row number
        key_to_row = {}

        key_column = headers[KEY_COL]

        for excel_row in range(2, ws.max_row + 1):
            key_value = ws.cell(
                excel_row,
                key_column
            ).value

            key_to_row[key_value] = excel_row

        # Colour mismatching cells red
        for key, trait in mismatch_cells:

            if (
                key in key_to_row
                and trait in headers
            ):

                cell = ws.cell(
                    row=key_to_row[key],
                    column=headers[trait]
                )

                cell.fill = red_fill
                cell.font = red_font

    # --------------------------------------------------------
    # General workbook formatting
    # --------------------------------------------------------

    for ws in wb.worksheets:

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        # Headers
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font

        # Sensible column widths
        for column_cells in ws.columns:

            column_letter = get_column_letter(
                column_cells[0].column
            )

            max_length = 0

            for cell in column_cells:

                if cell.value is not None:
                    max_length = max(
                        max_length,
                        len(str(cell.value))
                    )

            ws.column_dimensions[
                column_letter
            ].width = min(max(max_length + 2, 10), 45)

    # Percentage formatting
    for sheet_name in [
        "Row summary",
        "Trait summary",
    ]:

        ws = wb[sheet_name]

        for cell in ws[1]:

            if (
                cell.value
                and "%" in str(cell.value)
            ):

                for row in range(2, ws.max_row + 1):
                    ws.cell(
                        row,
                        cell.column
                    ).number_format = "0.0"

    wb.save(output_file)


# ============================================================
# RUN
# ============================================================

pred_df = pd.read_csv(PRED_FILE)
eval_df = pd.read_csv(EVAL_FILE)

(
    row_summary,
    trait_summary,
    differences,
    mismatch_cells,
) = evaluate_traits(pred_df, eval_df)


create_excel_report(
    pred_df,
    eval_df,
    row_summary,
    trait_summary,
    differences,
    mismatch_cells,
    OUTPUT_FILE,
)


print(f"Written to: {OUTPUT_FILE}")

print("\nPer-taxon accuracy:")
print(
    row_summary[
        [
            "taxon",
            "unmatched values",
            "value accuracy %",
        ]
    ]
    .sort_values("value accuracy %")
    .to_string(index=False)
)

print("\nPer-trait accuracy:")
print(
    trait_summary[
        [
            "trait",
            "unmatched values",
            "value accuracy %",
        ]
    ]
    .sort_values("value accuracy %")
    .to_string(index=False)
)

Written to: trait_evaluation.xlsx

Per-taxon accuracy:
               taxon  unmatched values  value accuracy %
       urtica dioica                29         32.558140
         betula nana                17         43.333333
       rumex acetosa                28         44.000000
phalaris arundinacea                10         50.000000
achillea millefolium                24         51.020408
    carex capillaris                 9         57.142857
eleocharis palustris                12         62.500000
hypochaeris radicata                14         64.102564
    valeriana dioica                11         68.571429
    stellaria alsine                 9         70.000000
    salix reticulata                13         70.454545
 vaccinium myrtillus                11         73.170732
   cuscuta epithymum                 5         76.190476
 andromeda polifolia                 5         82.142857
achillea millefolium                 8         83.333333

Per-trait accuracy:
            